In [274]:
import os
import pandas as pd
from datetime import datetime
import importlib

In [275]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [276]:
compact = False

In [277]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading 2024-08-30_22-21-59.parquet
Reading 2024-08-30_23-08-29.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4843967728,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.45420,0.013626,0,"[7, 1, 0, 0, 0]"
6139257088,4.843968e+09,[],"[96, 91]",0,"[4, 9]","[4, 9]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,5,0.45420,0.029523,0,"[7, 1, 0, 0, 0]"
4865770928,6.139257e+09,"[24, 20, 4]","[91, 91]",0,"[0, 0]","[9, 9]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,9,0.71980,0.064782,1,"[7, 11, 4, 1, 0]"
4843775744,4.865771e+09,"[24, 20, 4, 50]","[82, 82]",0,"[0, 0]","[18, 18]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,18,0.73890,0.133002,2,"[11, 7, 4, 0, 0]"
6141055568,NaN,[],"[114, 78]",0,"[4, 4]","[4, 4]","[False, True]","[False, False]",1,4,Tord,HumanPlayer,raise,4,0.64511,0.025804,0,"[12, 11, 0, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6073929200,6.073905e+09,[],"[50, 137]",0,"[4, 9]","[4, 9]","[True, True]","[False, False]",0,4,Alun,HumanPlayer,call,5,0.54259,0.035268,0,"[11, 1, 0, 0, 0]"
6073852832,6.073929e+09,"[17, 4, 9]","[45, 137]",0,"[0, 0]","[9, 9]","[False, False]","[False, False]",0,4,Alun,HumanPlayer,check,0,0.49540,0.044586,1,"[4, 11, 9, 1, 0]"
6073692016,6.073853e+09,"[17, 4, 9]","[45, 128]",0,"[0, 9]","[9, 18]","[True, True]","[False, False]",0,4,Alun,HumanPlayer,raise,30,0.49540,0.066879,1,"[4, 11, 9, 1, 0]"


In [278]:
raw_df.dtypes

prev_entry           float64
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

## Process data

In [279]:
df = Processor(raw_df).get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage
4843967728,c83b1495-0e33-4677-9297-83ee846f5133,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,2,0,0.45420,0.013626,preflop
6139257088,c83b1495-0e33-4677-9297-83ee846f5133,0,0,0,0,0,2,0,0,0,...,0,0,0,0,call,5,0,0.45420,0.029523,preflop
4865770928,c83b1495-0e33-4677-9297-83ee846f5133,0,0,0,0,0,7,0,0,0,...,1,0,0,0,raise,9,1,0.71980,0.064782,flop
4843775744,c83b1495-0e33-4677-9297-83ee846f5133,0,9,0,0,0,7,0,0,0,...,1,0,0,0,raise,18,1,0.73890,0.133002,turn
6141055568,645206a2-b235-44a0-a94a-8994727764fd,0,0,0,0,0,0,0,0,0,...,0,0,0,0,raise,4,0,0.64511,0.025804,preflop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6073929200,c94d5ccc-e0f2-497b-8be2-58dc2b4c8b0e,0,0,0,0,0,2,0,0,0,...,0,0,0,0,call,5,0,0.54259,0.035268,preflop
6073852832,c94d5ccc-e0f2-497b-8be2-58dc2b4c8b0e,0,0,0,0,0,7,0,0,0,...,1,0,0,0,check,0,0,0.49540,0.044586,flop
6073692016,c94d5ccc-e0f2-497b-8be2-58dc2b4c8b0e,0,0,0,0,0,7,0,0,0,...,1,0,0,0,raise,30,0,0.49540,0.066879,flop
6073730576,c94d5ccc-e0f2-497b-8be2-58dc2b4c8b0e,0,30,0,0,0,7,0,0,0,...,1,0,0,0,raise,15,0,0.48690,0.189891,turn


In [280]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [281]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [282]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [283]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_river,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage
4843967728,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,call,2,preflop
6139257088,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,call,5,preflop
4865770928,0,0,0,0,0,7,0,0,0,0,...,0,0,0,1,0,0,0,raise,9,flop
4843775744,0,9,0,0,0,7,0,0,0,0,...,0,0,0,1,0,0,0,raise,18,turn
6141055568,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,raise,4,preflop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6073929200,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,call,5,preflop
6073852832,0,0,0,0,0,7,0,0,0,0,...,0,0,0,1,0,0,0,check,0,flop
6073692016,0,0,0,0,0,7,0,0,0,0,...,0,0,0,1,0,0,0,raise,30,flop
6073730576,0,30,0,0,0,7,0,0,0,0,...,0,0,0,1,0,0,0,raise,15,turn


In [284]:
y

4843967728    0
6139257088    0
4865770928    1
4843775744    1
6141055568    0
             ..
6073929200    0
6073852832    0
6073692016    0
6073730576    0
6073720208    0
Name: excess_rank, Length: 210, dtype: int64

In [285]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [286]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (168, 33)
Test shape: (42, 33)


In [287]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7857142857142857


In [290]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7857142857142857
Mean goodness:  0.7407862809772711


,0,1,2,3,true,pred,correct,goodness
0,0.898921,0.062423,0.037517,1.138546e-03,0,0,True,0.898921
1,0.982346,0.013078,0.003600,9.762995e-04,0,0,True,0.982346
2,0.148984,0.809791,0.039840,1.384147e-03,1,1,True,0.809791
3,0.083922,0.708896,0.207034,1.483514e-04,1,1,True,0.708896
4,0.995756,0.002961,0.000622,6.605079e-04,0,0,True,0.995756
5,0.877907,0.101850,0.012309,7.934946e-03,5,0,False,0.000000
6,0.527924,0.323571,0.145450,3.054821e-03,5,0,False,0.000000
7,0.160103,0.801982,0.036417,1.498702e-03,5,1,False,0.000000
8,0.982346,0.013078,0.003600,9.762995e-04,0,0,True,0.982346
9,0.128539,0.813307,0.057649,5.049657e-04,0,1,False,0.128539
